# Agent Fundamentals- The Loop That Acts

**Module 11 · Lesson 11.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- What Is an Agent
- The Agent Loop
- Tool Selection & Calling
- ReAct: Reason, Then Act
- Termination & Reliability
- Memory: The Working Set

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" google-genai python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## From a Brain in a Jar to a Loop That Acts

> 💡 **Info**
>
> Ask an LLM “what is 14999 with 18 percent GST?” and it might *guess* 17698.82 - or hallucinate a number. Now give it a **calculator** and a **loop**: it THINKS “this needs arithmetic,” CALLS the calculator, OBSERVES `17698.82`, and ANSWERS - correct, every time. That loop - **think, act, observe, repeat** - is the entire difference between a chatbot and an agent, and it is about forty lines of Python. This lesson builds it from scratch, shows the real Anthropic `stop_reason` loop it maps onto, and teaches the judgement that separates good agent engineers from the hype: reliability compounds, so reach for an agent only when the task actually needs one.

### 🎯 What you will be able to do after this lesson

- **Build** the observe-think-act agent loop from scratch, and read the real Anthropic tool-use loop (`stop_reason == "tool_use"` → run tool → append `tool_result` → `end_turn`).

- **Compare** a chatbot vs a workflow vs an agent (who drives the control flow), blind tool calls vs ReAct, and an agent vs just calling the API.

- **Operate** the loop: select the right tool from a task, cap iterations, and reason about `p^N` reliability.

- **Evaluate** when an agent is warranted vs a workflow or a single call, using the autonomy dial and Anthropic’s “start simple” rule.

> 📦 **Info**
>
> ✅ Before you startThis builds on **10.1–10.2** (function calling and tool schemas - the model emits a typed call, your code runs it) and **10.5** (MCP: one way to expose tools). You should know what a tool call is and that a tool’s description is what the model reads to pick it. Comfort with basic Python control flow is enough.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🤖 **Analogy**
>
> **An LLM is a brain in a jar** - it can think, but it cannot DO anything. An agent gives the brain hands (tools: search, calculate, send email) and, crucially, a **loop**: think about the task → pick a tool → use it → look at the result → decide if done or try again. **Where it breaks down:** a real brain remembers; this one does not - the only memory is the running history you feed back each turn, which is why memory (Lesson 11.4) is engineered, not assumed.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“An agent is a chatbot with tools.”** A single tool call is not an agent. The **loop** is: the model decides an action, observes the result, and decides again - driving its own control flow.
> - **“More autonomy is always better.”** Reliability **compounds** (`p^N`): a tight workflow often beats an autonomous agent. Start simple; add agency only where flexibility earns its cost.

> 💡 **Info**
>
> ⚠️ Anti-patternReaching for an agent (or a framework) when the task is **deterministic or single-shot**. A one-shot translation should be one API call; a fixed nightly pipeline should be a workflow. Agents are for **open-ended, model-decided** paths - and they cost more and fail more, so they are the exception, not the default.

---

## 🎯 Concept 1: What Is an Agent

### What Is an Agent

Agent = the LLM drives the loop. Workflow = you drive. Chatbot = no loop at all. One dividing line.

The word “agent” is overused, so pin it down. Anthropic’s definition is the clearest: an **agent is an LLM autonomously using tools in a loop**; a **workflow** is an LLM (or several) running through a **path you fixed**, where you choose the steps and the model fills them in. The single dividing line is **who drives the control flow**: in a workflow, you do; in an agent, the LLM does. A useful shorthand is **Agent = LLM + Tools + Memory + Autonomy**, and the taxonomy runs chatbot (reactive Q&A) → copilot (suggests; the human acts) → agent (plans, acts, self-corrects). The test for “is this really an agent?”: does it call tools, do multi-step work, and decide when to stop?

> 🚗 **Analogy**
>
> It’s the difference between a **GPS, a friend, and a chauffeur**. A **workflow** is turn-by-turn directions you programmed. A **chatbot** is a friend who answers ‘how do I get to the airport?’ An **agent** is a chauffeur: you say ‘get me to the airport,’ and THEY choose the route, reroute around traffic, and stop when you arrive. The chauffeur drives; you only set the goal.

A fixed RAG pipeline retrieves, then always summarizes, in the same order every time. Agent or workflow?

**📝 `01_classify.py`** - *runnable*

In [ ]:
# WHO DRIVES THE CONTROL FLOW? - the one line between chatbot, workflow, and agent.
def classify(has_tools, llm_drives_control_flow, loops_on_results):
    if not has_tools and not loops_on_results:
        return "chatbot",  "reactive Q&A; no tools, no loop"
    if has_tools and not llm_drives_control_flow:
        return "workflow", "tools on a path YOU fixed; the LLM fills steps, you drive"
    if has_tools and llm_drives_control_flow and loops_on_results:
        return "agent",    "the LLM picks the next action and loops on the result"
    return "copilot", "suggests; the human decides and acts"

systems = [
    ("a plain support bot",       False, False, False),
    ("a fixed RAG pipeline",      True,  False, False),
    ("a tool-using research bot", True,  True,  True),
]
for name, tools, drives, loops in systems:
    kind, why = classify(tools, drives, loops)
    print(f"  {name:26s} -> {kind:8s} ({why})")

# Output:
#   a plain support bot        -> chatbot  (reactive Q&A; no tools, no loop)
#   a fixed RAG pipeline       -> workflow (tools on a path YOU fixed; the LLM fills steps, you drive)
#   a tool-using research bot  -> agent    (the LLM picks the next action and loops on the result)

- The one input that flips the label is `llm_drives_control_flow`.
- No tools + no loop → a **chatbot**; tools on a fixed path → a **workflow**.
- Tools + the LLM choosing the next action + looping on results → an **agent**.
- A copilot sits in between: it suggests, but a human decides and acts.

#### 💡 What just happened

⚡ What just happened? An agent is an LLM using tools **in a loop it drives**; a workflow is a path YOU drive. Tradeoff / when to use which: workflows are predictable and cheap for well-defined tasks; agents trade that for flexibility when the path must be decided at runtime. This distinction (Anthropic) is the spine of the whole module.

- Classify a system as chatbot, workflow, or agent
- Toggle who decides the next step: you or the LLM

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Agent Loop

### The Agent Loop

Observe -> think -> act, built from scratch. About forty lines, and every framework wraps it.

Here is the whole idea in code. The loop: **THINK** (the model looks at the task and the history and decides the next step), **ACT** (call the chosen tool with arguments), **OBSERVE** (feed the tool’s result back into the history), and repeat until the model gives a final answer. Below, a *scripted stand-in* plays the model’s THINK step so the loop runs without an API key - but the shape is exactly what a real agent does. Watch it solve a one-tool task (arithmetic) and a two-tool task (look up the refund policy, then the schedule) purely by looping on what it observes.

> 🍳 **Analogy**
>
> It’s a **cook tasting as they go**. Think (what does it need?) → act (add salt) → observe (taste) → think again, looping until the dish is right. A recipe card followed to the letter is a **workflow**; the taste-and-adjust loop is what makes a cook an **agent**.

The agent calls `search(refund)` and gets a policy. What does it do on the very next turn?

**📝 `02_agent_loop.py`** - *runnable*

In [ ]:
# THE AGENT LOOP FROM SCRATCH - observe -> think -> act, in ~20 lines (fake-LLM sim).
KB = {"genai": "GenAI course: 14999 INR, 146 hrs",
      "refund": "100% within 7 days, 50% within 30",
      "schedule": "Live class 7 PM IST daily"}
def search(q): return KB.get(q.lower().split()[0], "not found")
def calculate(expr): return round(eval(expr, {"__builtins__": {}}), 2)
TOOLS = {"search": search, "calculate": calculate}

def fake_llm(task, history):
    "A SCRIPTED stand-in for the LLM's THINK step (a real model decides this)."
    t = task.lower(); used = [h["tool"] + ":" + h["arg"] for h in history]
    if "gst" in t or "18%" in t:
        if "calculate:14999*1.18" not in used: return {"act": ("calculate", "14999*1.18")}
        return {"answer": f"14999 with 18% GST = {history[-1]['result']} INR"}
    if "refund" in t and ("class" in t or "live" in t):
        if "search:refund" not in used:   return {"act": ("search", "refund")}
        if "search:schedule" not in used: return {"act": ("search", "schedule")}
        return {"answer": f"Refund: {history[0]['result']}. Live class: {history[1]['result']}"}
    return {"answer": "I do not know."}

def agent(task, max_steps=5):
    history = []
    for step in range(max_steps):
        decision = fake_llm(task, history)                       # THINK
        if "answer" in decision:
            print(f"    step {step+1}: ANSWER -> {decision['answer']}"); return
        tool, arg = decision["act"]                              # ACT
        result = TOOLS[tool](arg)
        history.append({"tool": tool, "arg": arg, "result": result})   # OBSERVE
        print(f"    step {step+1}: ACT {tool}({arg}) -> OBSERVE {result}")

for task in ["What is 14999 with 18% GST?", "Refund policy and when is the live class?"]:
    print(f"  Task: {task}"); agent(task)

# Output:
#   Task: What is 14999 with 18% GST?
#     step 1: ACT calculate(14999*1.18) -> OBSERVE 17698.82
#     step 2: ANSWER -> 14999 with 18% GST = 17698.82 INR
#   Task: Refund policy and when is the live class?
#     step 1: ACT search(refund) -> OBSERVE 100% within 7 days, 50% within 30
#     step 2: ACT search(schedule) -> OBSERVE Live class 7 PM IST daily
#     step 3: ANSWER -> Refund: 100% within 7 days, 50% within 30. Live class: Live class 7 PM IST daily

- `fake_llm` is a scripted THINK step - a real model decides this from the task + history.
- For the GST task it ACTs once (calculate), OBSERVEs 17698.82, then ANSWERs.
- For the two-part task it loops: search(refund) → search(schedule) → combine into one answer.
- The `history` list threaded through every turn is the agent’s working memory (Step 6).

#### 💡 What just happened

⚡ What just happened? The agent loop is **think → act → observe**, repeating until the model answers - about twenty lines here. When to use a framework instead: only when you need what it adds (checkpoints, parallelism, multi-agent). Understand this loop and every framework in Lessons 11.2–11.5 becomes trivial.

- Step the loop think -> act -> observe on a task
- Watch the history grow until the agent answers

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Tool Selection & Calling

### Tool Selection & Calling

The model reads the tool descriptions, picks one, and fills its arguments. The real loop is driven by stop_reason.

How does the agent choose *which* tool? It reads each tool’s **description** (name + docstring + typed parameters - the 10.2 schema) and matches it to the task, then fills the arguments. The model can only pick from what the descriptions tell it, which is why a vague description leads to a wrong call. In the real Anthropic Messages API, this is structured, not text-parsing: the response comes back with **`stop_reason == "tool_use"`** and a `tool_use` block (name + input); your code runs the tool, appends a **`tool_result`** block, and loops - ending when `stop_reason` becomes **`end_turn`**. (OpenAI’s Agents SDK `Runner` and Google’s SDK do this loop for you.)

> 🔧 **Analogy**
>
> It’s a **mechanic reaching into a toolbox**. Given the job, they read the labels and grab the wrench - not the hammer. The agent reads the tool **descriptions** the same way; a mislabelled drawer (a vague description) is how you get the wrong tool.

A task says ‘what is 14999 * 1.18?’ Which tool should the model pick, and why?

**📝 `03_tool_selection.py`** - *runnable*

In [ ]:
# TOOL SELECTION - the model picks WHICH tool + args by reading the tool descriptions.
TOOL_DESCS = {"search": "look up course info, refunds, schedules",
              "calculate": "arithmetic like 14999*1.18",
              "send_email": "send an email to a user"}
def select_tool(task):
    t = task.lower()
    if any(w in t for w in ["*", "+", "gst", "%", "total"]): return ("calculate", "14999*1.18")
    if any(w in t for w in ["price", "refund", "schedule", "how much"]): return ("search", "genai")
    if "email" in t or "receipt" in t: return ("send_email", "asha@example.com")
    return (None, None)

for task in ["What is 14999 with 18% GST?", "How much is the GenAI course?", "Email the receipt to Asha"]:
    tool, arg = select_tool(task)
    print(f"  {task:38s} -> {tool}({arg!r})")

# Output:
#   What is 14999 with 18% GST?            -> calculate('14999*1.18')
#   How much is the GenAI course?          -> search('genai')
#   Email the receipt to Asha              -> send_email('asha@example.com')

- Each task is matched to a tool by the keywords its description advertises.
- Arithmetic → `calculate`; a lookup → `search`; a message → `send_email`.
- Pick `search` for a math task and it returns ‘not found’ - the wrong tool wastes a step.
- This matcher is a stand-in; a real model reads the full descriptions to choose.

**📝 `03b_real_loop.py`** - *Anthropic*

In [ ]:
# THE REAL LOOP - Anthropic Messages API tool use; stop_reason drives the while.
import anthropic
client = anthropic.Anthropic()                       # reads ANTHROPIC_API_KEY

TOOLS = [{"name": "calculate", "description": "Do arithmetic, e.g. 14999*1.18.",
          "input_schema": {"type": "object", "properties": {"expr": {"type": "string"}},
                           "required": ["expr"]}}]
def calculate(expr): return round(eval(expr, {"__builtins__": {}}), 2)

messages = [{"role": "user", "content": "What is 14999 with 18% GST?"}]
max_steps = 5                                        # cap iterations - the reliability rule from Step 5
for step in range(max_steps):
    resp = client.messages.create(model="claude-sonnet-4-6", max_tokens=1024,
                                  tools=TOOLS, messages=messages)
    if resp.stop_reason != "tool_use":               # not tool_use: stop. end_turn is clean; max_tokens/refusal land here too
        print(resp.content[0].text); break
    messages.append({"role": "assistant", "content": resp.content})
    results = []
    for block in resp.content:
        if block.type == "tool_use":                 # ACT: the model asked for a tool
            out = calculate(**block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(out)})
    messages.append({"role": "user", "content": results})   # OBSERVE: feed results back

# Output: (illustrative minimal example - needs `pip install anthropic` + a key; in production add
#   a request timeout= and wrap the call in try/except) the SAME think->act->observe loop as the sim
#   above, but the model decides and stop_reason (tool_use vs end_turn) drives it.

- The response’s `stop_reason` tells you what to do: `tool_use` means run a tool; `end_turn` means done.
- You append the assistant’s `tool_use` block, then a matching `tool_result` as a user message.
- The loop repeats that until `end_turn` - the exact think/act/observe loop, now model-driven - and `max_steps` caps it so a stuck agent cannot spin forever.
- Only `end_turn` is a clean finish; `max_tokens` means the reply was cut off and `refusal` means the model declined - treat those as stop-and-report, not another tool call.
- Tool use is two round trips: the model asks, you execute, the model answers.

#### 💡 What just happened

⚡ What just happened? The model selects a tool by reading its **description** and fills the arguments; a good schema (10.2) is what makes the pick correct. In production, **`stop_reason`** drives the loop (`tool_use` vs `end_turn`) - the same shape as the from-scratch sim. Tradeoff: a vague tool description is how the model picks the wrong tool, so a clear schema (10.2) is the highest-leverage fix; hand the loop to a framework once you have seen the raw version.

- Send a task and watch the model match a tool and fill args
- See a wrong pick vs a right pick, and the stop_reason loop

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: ReAct: Reason, Then Act

### ReAct: Reason, Then Act

Thought -> Action -> Observation. Writing the reasoning first is what makes tool use reliable.

Left to call tools blindly, a model grabs a familiar-looking one and often picks wrong. **ReAct** (Reason + Act) fixes this by interleaving reasoning and action: before each step the agent writes a **Thought** (“this is arithmetic, so I should calculate”), then takes an **Action** (call the tool), then records the **Observation** (the result), and repeats. That short reasoning step is cheap and it catches the wrong choice *before* the agent acts on it - which is why ReAct is the default orchestration for LLM agents. It is chain-of-thought (5.3) extended into the loop: reason, then act, then observe.

> 📝 **Analogy**
>
> It’s **showing your work on a maths exam**. Blurting an answer is error-prone; writing ‘this is a percentage problem, so I multiply’ first catches the slip before you commit. The **Thought** before the **Action** is that shown work.

Two agents face ‘what is 14999 with 18 percent GST?’ One calls a tool blindly, one writes a Thought first. Who is more reliable?

**📝 `04_react.py`** - *runnable*

In [ ]:
# ReAct - write a Thought BEFORE the Action; reasoning catches the wrong tool.
def blind(task):                       # no reasoning: grab a familiar tool
    return "search"
def react(task):                       # reason first, then pick
    thought = "The task is arithmetic, so I should CALCULATE, not search."
    return thought, "calculate"

task = "What is 14999 with 18% GST?"
print(f"  Task: {task}")
print(f"  BLIND -> Action: {blind(task)}  (wrong: search cannot do math -> 'not found')")
th, act = react(task)
print(f"  ReAct -> Thought: {th}")
print(f"        -> Action:  {act}  (right)")

# Output:
#   Task: What is 14999 with 18% GST?
#   BLIND -> Action: search  (wrong: search cannot do math -> 'not found')
#   ReAct -> Thought: The task is arithmetic, so I should CALCULATE, not search.
#         -> Action:  calculate  (right)

- The **blind** agent grabs `search` - a tool that cannot do math - and fails.
- The **ReAct** agent first writes a Thought that names the task as arithmetic.
- That reasoning leads it to the right Action: `calculate`.
- Thought → Action → Observation, repeating, is the ReAct pattern behind most agents.

#### 💡 What just happened

⚡ What just happened? ReAct interleaves **reasoning and acting**: a Thought before each Action catches the wrong tool before it runs. Tradeoff: reasoning spends a few extra tokens per step, but it buys reliability - usually worth it. Richer planning and reflection patterns build on this in Lesson 11.2.

- Build the Thought -> Action -> Observation trace
- Toggle reasoning off (blind) vs on (ReAct) and watch reliability

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Termination & Reliability

### Termination & Reliability

The loop stops on a final answer. But reliability compounds - p^N - so you cap iterations.

An agent stops naturally when the model returns a final answer instead of a tool call (in the API, `stop_reason` flips from `tool_use` to `end_turn`). But you also need a **hard cap** (`max_steps`), because of the single most important fact in agent engineering: **reliability compounds**. If each step succeeds with probability `p`, an `N`-step run succeeds with roughly `p^N` - so a 10-step agent whose steps are each 95% reliable succeeds only about **60%** of the time. Errors do not average out; they multiply. This is why Anthropic’s guidance is to *find the simplest solution and only add agency when needed*, and why production agents cap iterations and add human checkpoints where a mistake is costly (Lesson 11.10).

> 🏁 **Analogy**
>
> It’s a **relay race with N handoffs**. Each runner is excellent - 95% reliable - but drop the baton at any single handoff and the race is lost. Ten handoffs at 95% each win only about six races in ten. Fewer handoffs (fewer steps) win more races.

Each step of your agent is 95% reliable. Over 10 steps, roughly what is the end-to-end success rate?

**📝 `05_reliability.py`** - *runnable*

In [ ]:
# RELIABILITY COMPOUNDS - each step is ~95%, but the run multiplies (p^N).
def reliability(p, n): return round(p ** n, 3)
print("  per-step success p = 0.95:")
for n in [1, 3, 5, 10, 15]:
    bar = "#" * int(reliability(0.95, n) * 20)
    print(f"    {n:>2} steps -> {reliability(0.95, n):.2f}  {bar}")

# Output:
#   per-step success p = 0.95:
#      1 steps -> 0.95  ###################
#      3 steps -> 0.86  #################
#      5 steps -> 0.77  ###############
#     10 steps -> 0.60  ###########
#     15 steps -> 0.46  #########

- Per-step reliability is 0.95, but the run multiplies: `0.95 ** N`.
- At 10 steps it is already about 0.60; at 15, under 0.50.
- The bars make the compounding visible - each added step shortens the bar.
- Fix: a `max_steps` cap, natural termination, and checkpoints on costly steps.

#### 💡 What just happened

⚡ What just happened? An agent terminates on a final answer (`end_turn`), but reliability **compounds** (`p^N`), so you cap iterations. When to add a human: exactly where the cost of an autonomous mistake exceeds a quick review - the heart of Lesson 11.10. The takeaway: shorter agents are more reliable agents.

- Slide per-step reliability p and the step count N
- Watch p^N fall and set an iteration cap

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Memory: The Working Set

### Memory: The Working Set

The in-loop history is short-term memory. It grows every turn; deep memory systems come in 11.4.

Where does an agent “remember” things during a task? In the **history** the loop accumulates - the list of prior actions and observations that you feed back into the model each turn. That running history *is* the agent’s **short-term memory**, and it lives in the model’s **context window**. Two consequences: it grows every turn (so context and cost grow with it), and it is bounded by the window. When the history outgrows the window, you **summarize** old turns or move them to **long-term memory** - an external store (a file, a vector DB) the agent reads and writes across sessions. Modern models carry large windows, but effective recall degrades before the advertised limit, which is exactly why memory is engineered rather than assumed. The deep dive - episodic and semantic memory, vector stores, the CLAUDE.md pattern - is Lesson 11.4.

> 📋 **Analogy**
>
> It’s a **whiteboard during a meeting**. The running notes everyone works from are the short-term memory; when the board fills, you erase the stale items or file them in a cabinet (long-term storage). The agent’s history list is that whiteboard.

Your agent’s history grows past the model’s context window mid-task. What is the fix?

**📝 `06_memory.py`** - *runnable*

In [ ]:
# MEMORY - the in-loop history IS the short-term memory; it grows every turn.
def context_tokens(system, task, turns, per_turn=40):
    return system + task + per_turn * turns
sys_t, task_t = 200, 30
print("  history (and context, and cost) grows each turn:")
for turn in range(1, 6):
    print(f"    after turn {turn}: history={turn} entries, ~{context_tokens(sys_t, task_t, turn)} tokens")

# Output:
#   history (and context, and cost) grows each turn:
#     after turn 1: history=1 entries, ~270 tokens
#     after turn 2: history=2 entries, ~310 tokens
#     after turn 3: history=3 entries, ~350 tokens
#     after turn 4: history=4 entries, ~390 tokens
#     after turn 5: history=5 entries, ~430 tokens

- Each turn appends to `history`, so the context (and the bill) grows turn by turn.
- The `context_tokens` model shows the linear growth of the running history.
- That history IS the short-term memory the model reasons from - nothing persists by default.
- Past the window: summarize or externalize to long-term memory (the subject of Lesson 11.4).

#### 💡 What just happened

⚡ What just happened? An agent’s short-term memory is the **in-loop history** in the context window; long-term memory is an external store you engineer. When to externalize: once the history outgrows the window or must survive across sessions. Nothing is remembered automatically - which is why Lesson 11.4 is a whole lesson on memory.

- Run the loop and watch the history accumulate
- See the context window fill; summarize or externalize

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: When to Use an Agent (and When Not)

### When to Use an Agent (and When Not)

Deterministic or single-shot? Do not build an agent. Autonomy is a dial, not a switch. Frameworks + India.

The most valuable agent skill is knowing when NOT to build one. Reach for an agent only when the path is **open-ended and the model must decide it at runtime**. For a single deterministic operation, just call the API; for a single-shot request, one LLM call; for a fixed pipeline, a workflow. And autonomy is a **dial**, not a switch - think of the human’s role sliding from Operator → Collaborator (approves each action) → Consultant → Approver (yes/no on a full plan) → Observer (full autonomy); most production agents sit in the middle. The frameworks you will meet next - **OpenAI Agents SDK** (Agent / Runner / Handoffs / Guardrails), **LangGraph**, **CrewAI**, the **Claude Agent SDK**, **Google ADK** - all wrap the loop you just built; the depth is in Lessons 11.2–11.5. For Indic-language agents, **Sarvam** and **Bhashini** supply the models and speech tools.

> 🔧 **Analogy**
>
> It’s a **power drill versus a screwdriver**. For one screw, the screwdriver (a single API call) is faster; for an assembly line, a jig (a workflow); you reach for the cordless drill with a mind of its own (an agent) only when the job is unpredictable enough to need it.

A nightly job runs the same five fixed steps every time. Agent, workflow, or a single API call?

**📝 `07_decision.py`** - *runnable*

In [ ]:
# AGENT, WORKFLOW, OR JUST AN API CALL? - a decision function.
def decide(single_shot, deterministic, needs_runtime_flexibility):
    if single_shot: return "just call the API once (no loop needed)"
    if deterministic and not needs_runtime_flexibility: return "use a WORKFLOW (you fix the steps)"
    if needs_runtime_flexibility: return "use an AGENT (the LLM decides the path)"
    return "start with a workflow; add agency only if it is needed"

cases = [
    ("translate one sentence",              True,  True,  False),
    ("nightly ETL with fixed steps",        False, True,  False),
    ("research an open question via tools",  False, False, True),
]
for label, single, det, flex in cases:
    print(f"  {label:38s} -> {decide(single, det, flex)}")

# Output:
#   translate one sentence                 -> just call the API once (no loop needed)
#   nightly ETL with fixed steps           -> use a WORKFLOW (you fix the steps)
#   research an open question via tools    -> use an AGENT (the LLM decides the path)

- Single-shot → one API call; deterministic multi-step → a workflow; open-ended → an agent.
- The rule follows Anthropic’s guidance: start simple, add agency only when flexibility is worth it.
- ‘Needs runtime flexibility’ is the trigger - the path cannot be fixed in advance.
- Most tasks are NOT agents; that is a feature, not a failure.

**📝 `07b_one_flag_agent.py`** - *google-genai*

In [ ]:
# ONE-FLAG AGENT - the new google-genai SDK runs the loop for you (AFC on by default).
from google import genai
client = genai.Client()                              # reads GEMINI_API_KEY

def search_courses(topic: str) -> dict:
    "Search the Netsetos course catalog by topic."
    return {"genai": {"price": 14999}}.get(topic.lower(), {"error": "not found"})

resp = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="How much is the GenAI course?",
    config={"tools": [search_courses]},              # AFC runs the loop internally
)
print(resp.text)

# Output: (illustrative - needs `pip install google-genai` + a key) Gemini calls search_courses
#   for you and answers. Frameworks (OpenAI Agents SDK, LangGraph, CrewAI) wrap the same loop.
#   NOTE: the old `import google.generativeai` + enable_automatic_function_calling=True is deprecated.

- The new `google-genai` SDK runs the loop for you - automatic function calling is on by default.
- You pass the tools; Gemini decides, calls, observes, and answers internally.
- Every framework (OpenAI Agents SDK, LangGraph, CrewAI) wraps the same loop you built by hand.
- The old `import google.generativeai` + `enable_automatic_function_calling=True` is deprecated.

#### 💡 What just happened

⚡ What just happened? Use an agent only for **open-ended, model-decided** paths; a deterministic or single-shot task is an API call or a workflow. Autonomy is a dial (Operator → Observer), and most agents sit in the middle. When to use a framework: once you need what it adds - the loop itself you now understand cold. Deep dives: architectures (11.2), LangGraph (11.3), memory (11.4), multi-agent (11.5), human-in-the-loop (11.10), eval and security (11.11).

- Answer deterministic? single-shot? flexible?
- Watch it route to API call, workflow, or agent; slide the autonomy dial

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> An agent is an LLM using tools in a **loop it drives** - the one thing that separates it from a chatbot or a workflow (Step 1). The loop is **think → act → observe**, about forty lines, and every framework wraps it (Step 2). The model selects a tool by reading its **description**, and in production `stop_reason` drives the loop (Step 3). Writing a **Thought** before each Action - ReAct - makes tool use reliable (Step 4). But reliability **compounds** (`p^N`), so you cap iterations and checkpoint costly steps (Step 5). The agent’s working memory is the **in-loop history** in the context window (Step 6). And the sharpest skill is knowing when NOT to build one: deterministic or single-shot tasks are API calls or workflows (Step 7). You now have the loop the entire rest of Module 11 builds on.

> 📦 **Info**
>
> ➡️ Where this goes nextYou have the fundamental loop. Now Module 11 builds up: agent **architectures** (planning, reflection) in Lesson 11.2, **LangGraph** in Lesson 11.3, **memory systems** in Lesson 11.4, **multi-agent** coordination in Lesson 11.5, **human-in-the-loop** and safety in Lesson 11.10, and **evaluation and security** in Lesson 11.11. The reference is Anthropic’s [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) and the [tool-use loop docs](https://platform.claude.com/docs/en/agents-and-tools/tool-use/implement-tool-use).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Agent Fundamentals- The Loop That Acts**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.1.html` - regenerate this notebook whenever the source HTML is updated.*
